### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** DQN was a watershed moment — it proved deep learning + RL could solve complex tasks from raw observations. The two stabilization tricks (replay buffer and target network) appear EVERYWHERE in modern deep RL, including PPO and the RL step of RLHF.

**Mental Model:** Imagine studying for an exam by only reading the textbook in order, page by page, and updating your notes after every page. Bad idea — you'd keep overwriting what you learned early. Experience Replay is like shuffling your flashcards randomly. Target Network is like having a stable answer key that you only update weekly, instead of changing it every minute.

**Common Junior Pitfall:** Training DQN with a learning rate that's too high. DQN is extremely sensitive to hyperparameters — use Adam with lr=1e-4 and be patient. Training for 1M steps is normal.

---

## 📑 Table of Contents

* [🎓 Socratic Deep Check](#-socratic-deep-check)
* [The Jump: Tables → Neural Networks](#the-jump-tables-neural-networks)
  * [🎓 Junior to Senior: Concept Bridge](#-junior-to-senior-concept-bridge)
* [1 · Why Naive Deep Q-Learning Fails](#1-why-naive-deep-q-learning-fails)
  * [Problem 1: Correlated Data](#problem-1-correlated-data)
  * [Problem 2: Moving Target](#problem-2-moving-target)
* [2 · Experience Replay (Trick #1)](#2-experience-replay-trick-1)
* [3 · Target Network (Trick #2)](#3-target-network-trick-2)
* [4 · Metric Tracking in Deep RL](#4-metric-tracking-in-deep-rl)
  * [Why Logging is Non-Negotiable](#why-logging-is-non-negotiable)
* [DQN Variants (Post-2013 Improvements)](#dqn-variants-post-2013-improvements)
  * [Prioritized Experience Replay (PER)](#prioritized-experience-replay-per)
  * [🎓 DEEP QUESTION ANSWERED](#-deep-question-answered)
* [✅ Knowledge Check](#-knowledge-check)
  * [Q1: Why gradient clipping?](#q1-why-gradient-clipping)
  * [Q2: DQN vs Q-table](#q2-dqn-vs-q-table)
* [🔨 Practical Practice](#-practical-practice)
  * [Exercise 1: Double DQN](#exercise-1-double-dqn)
  * [Exercise 2: Replay Buffer Analysis](#exercise-2-replay-buffer-analysis)
  * [Exercise 3: Gym CartPole](#exercise-3-gym-cartpole)

---


In [ ]:
!pip install -q torch numpy matplotlib gymnasium tensorboard

## 1 · Why Naive Deep Q-Learning Fails

If you just replace the Q-table with a neural network and train with gradient descent, it **diverges**. Two fatal problems:

### Problem 1: Correlated Data
Consecutive experiences are highly correlated (state $s_t$ is very similar to $s_{t+1}$). Neural networks trained on correlated data overfit to recent experiences and forget earlier lessons.

```
Frame 100: agent is in room A  → learns about room A
Frame 101: still in room A     → overwrites other knowledge
Frame 102: still in room A     → catastrophic forgetting of room B
```

### Problem 2: Moving Target
The TD target uses $\max_{a'} Q_\theta(s', a')$ — but we're updating the SAME network $Q_\theta$! The target moves as we learn, causing oscillations.

```
Update: Q(s,a) → target = r + γ max Q(s',a')  [target uses SAME network]
After update: Q changes → target changes → Q changes → target changes → 💥
```

---
## 2 · Experience Replay (Trick #1)

**Solution to correlated data:** Store experiences in a buffer, sample RANDOMLY during training.

```
Agent plays → stores (s, a, r, s', done) → Replay Buffer (100K experiences)

Training → randomly sample batch of 32 → train on random, decorrelated data
```

This breaks temporal correlations and allows the network to re-learn from past experiences.

In [ ]:
import numpy as np
from collections import deque
import random

class ReplayBuffer:
    """Experience Replay Buffer.
    
    Stores (state, action, reward, next_state, done) tuples.
    Random sampling breaks temporal correlation between experiences.
    
    This same concept appears in other contexts:
    - DPO training (NB12): shuffle preference pairs
    - Standard ML (NB10): shuffle training data each epoch
    """
    
    def __init__(self, capacity=100000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size=32):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards, dtype=np.float32),
                np.array(next_states), np.array(dones, dtype=np.float32))
    
    def __len__(self):
        return len(self.buffer)

# Demo
buffer = ReplayBuffer(capacity=10000)

# Fill with fake experiences
for i in range(500):
    s = np.random.randn(4)       # 4-dim state
    a = np.random.randint(2)     # binary action
    r = np.random.randn()        # random reward
    s_next = np.random.randn(4)  # next state
    done = random.random() < 0.1 # 10% chance of episode end
    buffer.push(s, a, r, s_next, done)

# Sample a random batch
states, actions, rewards, next_states, dones = buffer.sample(32)
print(f'Replay Buffer: {len(buffer)} experiences stored')
print(f'Sampled batch: {states.shape[0]} experiences')
print(f'  States shape:  {states.shape}')
print(f'  Actions:       {actions[:8]}')
print(f'  Rewards:       {rewards[:8].round(2)}')
print(f'\nKey insight: batch is RANDOM, not sequential.')
print(f'This breaks correlation — experiences from different episodes are mixed.')

---
## 3 · Target Network (Trick #2)

**Solution to moving target:** Use a SEPARATE, slowly-updated network for computing TD targets.

```
Online Network (θ):   updated every step with gradient descent
Target Network (θ⁻):  COPY of online network, updated every N steps

TD target = r + γ max_a' Q_θ⁻(s', a')    ← uses FROZEN target network
Loss = (Q_θ(s,a) - TD target)²             ← updates ONLINE network only
Every N steps: θ⁻ ← θ                     ← sync target to online
```

This makes the target STABLE for N steps, preventing the "chasing its own tail" problem.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

class DQN(nn.Module):
    """Deep Q-Network.
    
    Input: state vector (4 dimensions for CartPole)
    Output: Q-value for each action
    """
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )
    
    def forward(self, x):
        return self.net(x)

class DQNAgent:
    """Complete DQN Agent with Experience Replay, Target Network, and optional Double DQN."""
    
    def __init__(self, state_dim, action_dim, lr=1e-4, gamma=0.99,
                 epsilon=1.0, buffer_size=100000, batch_size=64,
                 target_update_freq=1000, use_ddqn=False):
        
        self.action_dim = action_dim
        self.gamma = gamma
        self.epsilon = epsilon
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self.use_ddqn = use_ddqn
        self.step_count = 0
        
        # Two networks: online (learns) + target (stable)
        self.online_net = DQN(state_dim, action_dim)
        self.target_net = DQN(state_dim, action_dim)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()
        
        self.optimizer = optim.Adam(self.online_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer(buffer_size)
    
    def choose_action(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.action_dim)
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0)
            q_values = self.online_net(state_t)
            return q_values.argmax(dim=1).item()
    
    def train_step(self):
        if len(self.buffer) < self.batch_size:
            return 0  # Not enough data yet
        
        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)
        
        states_t = torch.FloatTensor(states)
        actions_t = torch.LongTensor(actions)
        rewards_t = torch.FloatTensor(rewards)
        next_states_t = torch.FloatTensor(next_states)
        dones_t = torch.FloatTensor(dones)
        
        current_q = self.online_net(states_t).gather(1, actions_t.unsqueeze(1)).squeeze(1)
        
        with torch.no_grad():
            if self.use_ddqn:
                # DDQN: Online net selects action, Target net evaluates action
                next_actions = self.online_net(next_states_t).argmax(dim=1)
                next_q = self.target_net(next_states_t).gather(1, next_actions.unsqueeze(1)).squeeze(1)
            else:
                # Standard DQN: Target net selects and evaluates action
                next_q = self.target_net(next_states_t).max(dim=1)[0]
                
            td_target = rewards_t + self.gamma * next_q * (1 - dones_t)
        
        loss = nn.functional.mse_loss(current_q, td_target)
        
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.online_net.parameters(), max_norm=1.0)
        self.optimizer.step()
        
        self.step_count += 1
        if self.step_count % self.target_update_freq == 0:
            self.target_net.load_state_dict(self.online_net.state_dict())
        
        return loss.item()

# Demo: Architecture summary
agent = DQNAgent(state_dim=4, action_dim=2, use_ddqn=True) # Defaults to DDQN
total_params = sum(p.numel() for p in agent.online_net.parameters())

print('DQN Architecture:')
print(f'  Online Network:  {total_params:,} parameters')
print(f'  Target Network:  {total_params:,} parameters (frozen copy)')
print(f'  Replay Buffer:   {agent.buffer.buffer.maxlen:,} capacity')
print(f'  Target sync:     every {agent.target_update_freq} steps')


---
## 4 · Metric Tracking in Deep RL

### Why Logging is Non-Negotiable

Unlike supervised learning where loss decreases predictably, RL training is highly volatile. The agent generates its own data (which is non-stationary) and bootstraps its targets (creating "moving goalposts"). 

Because of this extreme variance:
1. **Loss is often meaningless:** A rising loss *can* mean the agent is discovering high-reward states (which have larger TD targets).
2. **You must track returns:** Average episode reward is the true measure of performance.
3. **You must track entropy/epsilon:** To ensure exploration doesn't die too early.
4. **You must track Max Q:** To ensure values aren't exploding to infinity (a common bug in DQN).

We'll use PyTorch's native `SummaryWriter` (TensorBoard) to track these metrics during our CartPole experiment.

In [ ]:
import gymnasium as gym
from torch.utils.tensorboard import SummaryWriter
import datetime
import os

# Setup TensorBoard writer
os.makedirs('runs', exist_ok=True)
run_name = f'dqn_cartpole_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}'
writer = SummaryWriter(f'runs/{run_name}')

# Construct real environment
env = gym.make('CartPole-v1')
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

# Instantiate our DDQN agent
agent = DQNAgent(state_dim=state_dim, action_dim=action_dim, lr=5e-4, gamma=0.99,
                 epsilon=1.0, batch_size=64, target_update_freq=200, use_ddqn=True) # DDQN enabled!

episode_rewards = []

for episode in range(300):
    state, _ = env.reset()
    total_reward = 0
    episode_loss = 0
    loss_steps = 0
    
    while True:
        action = agent.choose_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        agent.buffer.push(state, action, reward, next_state, done)
        loss = agent.train_step()
        if loss > 0:
            episode_loss += loss
            loss_steps += 1
            
        total_reward += reward
        state = next_state
        
        if done:
            break
    
    episode_rewards.append(total_reward)
    agent.epsilon = max(0.01, agent.epsilon * 0.995)
    
    # --- Log to TensorBoard ---
    writer.add_scalar('Train/Reward', total_reward, episode)
    writer.add_scalar('Train/Epsilon', agent.epsilon, episode)
    if loss_steps > 0:
        writer.add_scalar('Train/Avg_Loss', episode_loss / loss_steps, episode)
        
writer.close()

# Plotting local fallback
window = 20
smoothed = [np.mean(episode_rewards[max(0,i-window):i+1]) for i in range(len(episode_rewards))]

plt.figure(figsize=(10, 4))
plt.plot(smoothed, lw=2, color='steelblue', label='Smoothed DDQN')
plt.axhline(y=500, color='green', linestyle='--', alpha=0.7, label='Max (500 steps)') # CartPole-v1 max is 500
plt.xlabel('Episode')
plt.ylabel('Score (smoothed)')
plt.title('DDQN Learning: CartPole-v1')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('dqn_cartpole.png')
plt.show()

print(f'Early episodes: {np.mean(episode_rewards[:20]):.0f} score')
print(f'Late episodes:  {np.mean(episode_rewards[-20:]):.0f} score')
print(f'\nThe agent learned to consistently balance the pole!')
print(f'To view detailed dashboard: Run `tensorboard --logdir runs` in your terminal.')


---
## DQN Variants (Post-2013 Improvements)

| Variant | Year | Improvement | Key Idea |
|---------|------|-------------|----------|
| **Double DQN** | 2015 | Reduces overestimation | Use online net to SELECT action, target net to EVALUATE |
| **Dueling DQN** | 2016 | Better value estimation | Split Q into V(s) + A(s,a) |
| **Prioritized Replay** | 2016 | Better sample efficiency | Replay important experiences more often |
| **Rainbow** | 2017 | Combines all of the above | 6 improvements stacked together |

### Prioritized Experience Replay (PER)
Instead of uniformly sampling from the replay buffer, PER samples experiences proportional to how "surprising" they are. Surprisingly bad/good estimates have a large TD Error $\delta_i$.

1. **Priority Score:** Compute absolute TD error plus a small constant to ensure non-zero probability:
$$p_i = |\delta_i| + \epsilon$$
2. **Sampling Probability:** Convert scores to probabilities, controlled by factor $\alpha$ (0 = uniform, 1 = fully prioritized):
$$P(i) = \frac{p_i^\alpha}{\sum_k p_k^\alpha}$$
3. **Importance Sampling Weights:** Because prioritizing biases the data distribution, we must correct the gradient updates using importance sampling weights, parameterized by $\beta$:
$$w_i = \left( \frac{1}{N} \cdot \frac{1}{P(i)} \right)^\beta$$

### 🎓 DEEP QUESTION ANSWERED

**Q:** *What TWO tricks did DeepMind use to stabilize training?*

**A:**
1. **Experience Replay**: Store experiences in a buffer, sample randomly to break temporal correlations. This turns an RL problem into something closer to supervised learning (i.i.d. mini-batches).
2. **Target Network**: Use a frozen copy of the network for computing TD targets. Sync periodically. This stabilizes the target, preventing the "moving goalposts" problem.

---

## ✅ Knowledge Check

### Q1: Why gradient clipping?
We used `clip_grad_norm_(params, 1.0)` in the DQN training. Why is this important in RL but less critical in supervised learning?

<details><summary>👉 Answer</summary>

RL gradients are much noisier than supervised learning because: (1) the target itself is an estimate (bootstrapping), (2) the policy is changing (non-stationary data), (3) rewards can be sparse and high-variance. Without clipping, a single bad batch can produce a massive gradient that destroys the network. Clipping caps the gradient magnitude, keeping training stable.
</details>

### Q2: DQN vs Q-table
A DQN with a 2-layer network has 10K parameters. Is this more or less than a Q-table for Atari?

<details><summary>👉 Answer</summary>

MUCH less. Atari has ~10^67 possible screen configurations. A Q-table with 18 actions would need 10^67 × 18 entries — more than atoms in the universe. A 10K-parameter DQN generalizes across similar states (similar screenshots have similar Q-values), making it feasible.
</details>

---

## 🔨 Practical Practice

### Exercise 1: Double DQN
Implement Double DQN: use the online network to SELECT the best action, but the target network to EVALUATE it. Compare Q-value estimates to vanilla DQN.

### Exercise 2: Replay Buffer Analysis
Track TD errors during training. Implement Prioritized Experience Replay: sample experiences with probability proportional to their TD error (bigger errors = more to learn).

### Exercise 3: Gym CartPole
Install gymnasium and train DQN on the real CartPole-v1 environment. Target: 200+ average reward over 100 episodes.

---

**Next →** RL 04: Policy Gradient Methods (REINFORCE)